In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [2]:
df=pd.read_csv(r'C:\Users\johnf\forks\Statistical-Learning-Lab-Test\ai4i2020.csv')
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [4]:
all_tar=['Machine failure','TWF','HDF','PWF','OSF','RNF']
df=df.drop(columns=['UDI','Product ID'],inplace=False)

In [5]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16)
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )
    
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

In [8]:
def train_and_eval_autoenc(df,target_col):
    lis=all_tar.copy()
    lis.remove(target_col)
    temp=df.drop(columns=lis,inplace=False)
    cat_col = ["Type"]
    num_cols = [col for col in temp.columns if col not in cat_col + [target_col]]
    train_df, test_df = train_test_split(temp, test_size=0.2, random_state=42, stratify=temp[target_col])
    train_df = train_df[train_df[target_col] == 0]

    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(sparse_output=False), cat_col)
    ])

    X_train = preprocessor.fit_transform(train_df.drop(columns=[target_col]))
    X_test = preprocessor.transform(test_df.drop(columns=[target_col]))

    y_test = test_df[target_col].values

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_tensor), batch_size=64, shuffle=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = Autoencoder(input_dim=X_train.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    epochs = 50

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for (batch,) in train_loader:
            batch = batch.to(device)
            
            optimizer.zero_grad()
            recon = model(batch)
            loss = criterion(recon, batch)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        # print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

    model.eval()

    with torch.no_grad():
        X_test_device = X_test_tensor.to(device)
        recon = model(X_test_device)
        
        # MSE per sample
        errors = torch.mean((recon - X_test_device) ** 2, dim=1).cpu().numpy()

    threshold = np.percentile(errors, 95)

    with torch.no_grad():
        train_recon = model(X_train_tensor.to(device))
        train_errors = torch.mean((train_recon - X_train_tensor.to(device))**2, dim=1).cpu().numpy()

    threshold = np.percentile(train_errors, 95)

    y_pred = (errors > threshold).astype(int)

    from sklearn.metrics import classification_report, roc_auc_score

    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, errors))

In [9]:
train_and_eval_autoenc(df,'Machine failure')

              precision    recall  f1-score   support

           0       0.97      0.94      0.96      1932
           1       0.15      0.28      0.19        68

    accuracy                           0.92      2000
   macro avg       0.56      0.61      0.57      2000
weighted avg       0.95      0.92      0.93      2000

ROC-AUC: 0.7071991840214347


In [10]:
for tar in all_tar:
    print("Target Name: ",tar)
    train_and_eval_autoenc(df,tar)
    print("==================================================================================")
    print()

Target Name:  Machine failure
              precision    recall  f1-score   support

           0       0.99      0.95      0.97      1932
           1       0.31      0.59      0.41        68

    accuracy                           0.94      2000
   macro avg       0.65      0.77      0.69      2000
weighted avg       0.96      0.94      0.95      2000

ROC-AUC: 0.8621666057727438

Target Name:  TWF
              precision    recall  f1-score   support

           0       1.00      0.94      0.97      1991
           1       0.00      0.00      0.00         9

    accuracy                           0.94      2000
   macro avg       0.50      0.47      0.48      2000
weighted avg       0.99      0.94      0.96      2000

ROC-AUC: 0.735866956861432

Target Name:  HDF
              precision    recall  f1-score   support

           0       0.99      0.94      0.97      1977
           1       0.04      0.22      0.07        23

    accuracy                           0.93      2000
   ma